# Rebuilding the Social Engine — Round 2

**Team SE7EN** · Tanmay Singh · Panshul Arora
Data Vortex A'26 · Round 2 · NLP comprehension layer

---

This notebook walks the whole pipeline end to end on Dataset 2: audit the labels,
preprocess, build features, select a model, evaluate honestly, analyse the errors.

Every cell below has been executed; the outputs are what the code produced.
The same steps run headless with `python round2/run_round2.py`.

**Contents**

1. Load and profile the corpus
2. Two data-quality findings that changed the design
3. Preprocessing
4. Features
5. Model selection under grouped cross-validation
6. Held-out evaluation
7. Error analysis
8. Inference

In [1]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")

SRC = Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path("round2/src")
sys.path.insert(0, str(SRC.resolve()))

import numpy as np, pandas as pd
import sklearn
print("python     ", sys.version.split()[0])
print("scikit-learn", sklearn.__version__)
print("pandas     ", pd.__version__)

python      3.11.0
scikit-learn 1.5.1
pandas      2.2.2


## 1. Load and profile the corpus

In [2]:
from dataio import load, sha256, duplicate_report

df = load()
print("rows:", len(df), " sha256:", sha256()[:24], "...")
df.head(4)[["text_id", "post_text", "sentiment_label", "topic_category"]]

rows:

 9000  sha256: a866d9fd9d5b6e5559467fd8 ...


,text_id,post_text,sentiment_label,topic_category
0,TXT_00001,some1 come wait in line with me Thursday at 3@...,Negative,Community_Discussion
1,TXT_00002,Mancity strikers Aguero and Dzeko you better t...,Positive,Community_Discussion
2,TXT_00003,Would you like to join us at our annual gala a...,Positive,Community_Discussion
3,TXT_00004,"""Sicily may form grand coalition, Italian UDC ...",Neutral,Community_Discussion


In [3]:
print("sentiment_label")
print(df.sentiment_label.value_counts().to_string())
print()
print("topic_category")
print(df.topic_category.value_counts().to_string())
print()
words = df.post_text.str.split().str.len()
print(f"post length: median {words.median():.0f} words, "
      f"range {words.min()}-{words.max()}; no nulls anywhere: {df.isna().sum().sum() == 0}")

sentiment_label
sentiment_label
Negative    3000
Positive    3000
Neutral     3000

topic_category
topic_category
Community_Discussion    7752
Technical_Issues         815
Feature_Feedback         297
Account_Security         136

post length: median 20 words, range 4-35; no nulls anywhere: True


Sentiment is exactly balanced at 3,000 per class. Topic is not: 86% of the
corpus is `Community_Discussion` and `Account_Security` has 136 posts. That 57:1 skew
is why **macro-F1** leads every table in this submission — a model that only ever says
`Community_Discussion` would score 0.861 accuracy and be useless.

## 2. Two data-quality findings that changed the design

### 2.1 — 1,100 rows are duplicate posts

In [4]:
dup = duplicate_report(df)
for k, v in dup.items():
    print(f"{k:32s} {v}")

n_rows                           9000
n_unique_texts                   7900
n_duplicate_rows                 1100
n_texts_repeated                 987
max_repeats                      5
label_conflicts_among_repeats    {'sentiment': 0, 'topic': 0}


Every repeat carries an identical label, so the duplicates are not annotation
noise — they are the same post counted twice. Harmless for training, **dangerous for
evaluation**: a random row split puts copies of one post on both sides and grades the
model on memorised strings.

So we split on the *unique post*, never on the row — and we measure what the shortcut
would have been worth (Section 5).

### 2.2 — `topic_category` is generated by a substring rule

A character-ngram model beat a word-ngram model on `topic_category` by 23 macro-F1
points. Topics live in words, so that ordering is backwards. The top character
features said why: `ban`, `app`, `ui`, `mode` — substrings, not words.

`audit_labels.py` mines the rule from scratch and replays it.

In [5]:
from audit_labels import RECOVERED_RULE, PRIORITY, DEFAULT_CLASS, apply_rule

predicted = apply_rule(df.post_text)
fidelity = (predicted == df.topic_category).mean()
print(f"fidelity: {fidelity:.4f}  ({(predicted == df.topic_category).sum()} / {len(df)} rows)")
print()
for cls in PRIORITY:
    print(f"  {cls:22s} <- any of {RECOVERED_RULE[cls]}")
print(f"  {DEFAULT_CLASS:22s} <- (everything else)")

fidelity: 1.0000  (9000 / 9000 rows)

  Technical_Issues       <- any of ['app', 'down', 'update', 'crash', 'screen', 'slow', 'bug', 'glitch']
  Account_Security       <- any of ['ban', 'account', 'suspend', 'hack', 'password']
  Feature_Feedback       <- any of ['ui', 'mode', 'feature', 'ugly', 'design', 'button']
  Community_Discussion   <- (everything else)


**9,000 of 9,000 rows, exactly.** `topic_category` is not an annotation; it is a
case-insensitive substring switch over the raw post.

Because it matches substrings rather than words, the damage is easy to see:

In [6]:
probe = [
    "That Janet Jackson. Sometimes she just gets me. Happy Friday friends!",
    "Kenneth Downing Jr., guitarist in the British heavy metal band JUDAS PRIEST",
    "I use to love that song in the 8th grade. Trina was my role model lmfao",
    "Jurassic Park is screening at the Actors Playhouse this weekend for free",
]
for text in probe:
    print(f"{apply_rule(pd.Series([text]))[0]:22s} <- {text[:68]}")

Technical_Issues       <- That Janet Jackson. Sometimes she just gets me. Happy Friday friends
Technical_Issues       <- Kenneth Downing Jr., guitarist in the British heavy metal band JUDAS
Feature_Feedback       <- I use to love that song in the 8th grade. Trina was my role model lm
Technical_Issues       <- Jurassic Park is screening at the Actors Playhouse this weekend for 


Three consequences, and they shape everything after this cell:

1. The **Bayes error of the topic task is exactly zero** — the label is a deterministic
   function of the input. A perfect score is attainable and meaningless.
2. A topic model is rewarded for **recovering substrings**, not topics.
3. Roughly **one post in eight** carries a topic a human would call wrong.

We report this rather than harvesting the free 1.000. Sentiment shows no such
structure — cue words are graded ("love" is 81% Positive, not 100%), which is what
human annotation looks like.

In [7]:
for kw in ["love", "hate", "happy", "worst", ":)", ":("]:
    hit = df.post_text.str.lower().str.contains(kw, regex=False)
    share = df.sentiment_label[hit].value_counts(normalize=True).round(2).to_dict()
    print(f"{kw:8s} n={hit.sum():4d}  {share}")

love     n= 253  {'Positive': 0.81, 'Neutral': 0.1, 'Negative': 0.09}
hate     n=  82  {'Negative': 0.83, 'Positive': 0.1, 'Neutral': 0.07}
happy    n= 152  {'Positive': 0.84, 'Negative': 0.09, 'Neutral': 0.07}
worst    n=  43  {'Negative': 0.84, 'Neutral': 0.14, 'Positive': 0.02}
:)       n= 137  {'Positive': 0.83, 'Neutral': 0.11, 'Negative': 0.06}


:(       n=  85  {'Negative': 0.85, 'Positive': 0.08, 'Neutral': 0.07}


## 3. Preprocessing

The corpus carries damage from a bad serialisation round-trip (`\u002c` escapes, leaked
CSV quoting) on top of ordinary social noise. Normalisation repairs that and
canonicalises what is noise for a bag-of-ngrams model, while **preserving** what an
annotator actually used — emoticons, punctuation runs, hashtag bodies.

Negation scope is marked so that *"not a good day"* stops sharing every unigram with
*"a good day"*.

In [8]:
from preprocess import normalise, prepare

demo = [
    "I'm sooooo happy :) #GoodFriday @user http://t.co/x",
    "not a good day at all!!!",
    'Dwight Howard gets his 4th career 30-point\\u002c 10-rebound game.\\" Too bad it comes in a loss',
]
for text in demo:
    print("raw :", text)
    print("prep:", prepare(text))
    print()

raw : I'm sooooo happy :) #GoodFriday @user http://t.co/x
prep: i'm soo happy emotesmile # good friday usertoken urltoken

raw : not a good day at all!!!
prep: not a_neg good_neg day_neg at_neg all!!

raw : Dwight Howard gets his 4th career 30-point\u002c 10-rebound game.\" Too bad it comes in a loss
prep: dwight howard gets his 4th career numtoken -point, numtoken -rebound game." too bad it comes in a loss



What we deliberately **did not** do: no stop-word removal (`not`, `no`, `but` are all
on standard stop lists and all three are polarity-bearing), no stemming (it destroys
the substring evidence the topic task runs on), no external sentiment lexicon.

## 4. Features

Three complementary views, concatenated into one sparse matrix:

| block | what | why |
|---|---|---|
| `word` | TF-IDF 1–2 grams over the normalised text | lexical polarity, topical content |
| `char` | TF-IDF character n-grams inside word boundaries | misspellings, elongation, hashtag compounds — and substrings |
| `surface` | 11 counts from the **raw** string | the cues normalisation is about to destroy |

In [9]:
from features import build_features, SurfaceStats, STAT_NAMES

union = build_features()
X = union.fit_transform(df.post_text[:2000])
print("feature matrix:", X.shape, f"({X.nnz / X.shape[0]:.0f} non-zeros per post)")
print()
print("surface block:")
print(pd.DataFrame(SurfaceStats().transform(df.post_text[:3].tolist()),
                   columns=STAT_NAMES).round(2).to_string(index=False))

feature matrix: (2000, 25149) (209 non-zeros per post)

surface block:
 n_chars  n_words  n_exclaim  n_question  n_punct_run  n_allcaps  n_elong  n_hashtag  n_mention  emote_pos  emote_neg
    4.38     2.83        0.0         0.0          0.0        0.0      0.0        0.0        1.0        0.0        0.0
    4.32     2.48        1.0         0.0          0.0        0.0      0.0        0.0        0.0        0.0        0.0
    4.90     3.26        1.0         1.0          0.0        0.0      0.0        0.0        0.0        0.0        0.0


## 5. Model selection under grouped cross-validation

Eight candidates per task, scored with 5-fold `StratifiedGroupKFold` — stratified on
the label, grouped on the post, so no fold is ever graded on text it trained on.
Vectorisers are fitted **inside** each fold.

The list is an ablation: feature blocks are added one at a time with the classifier
held fixed, then the model family is varied with the features held fixed.

In [10]:
from config import REPORTS

for task in ["sentiment", "topic"]:
    t = pd.read_csv(REPORTS / f"model_comparison_{task}.csv")
    print(f"=== {task} " + "=" * 52)
    print(t[["model", "cv_f1_macro", "cv_f1_macro_std", "cv_accuracy"]]
          .round(4).to_string(index=False))
    print()

=== sentiment ====================================================
                                               model  cv_f1_macro  cv_f1_macro_std  cv_accuracy
           word + char 3-5gram + surface + LinearSVC       0.6227           0.0125       0.6224
  word + char 3-5gram + surface + LogisticRegression       0.6224           0.0184       0.6214
                     word + char 3-5gram + LinearSVC       0.6178           0.0120       0.6178
  word + char 3-5gram + surface + soft-vote ensemble       0.6165           0.0176       0.6178
                            char 3-5gram + LinearSVC       0.6055           0.0157       0.6063
                  word + char 3-5gram + ComplementNB       0.5958           0.0182       0.6002
word + char 3-5gram + surface + SGD (modified huber)       0.5927           0.0135       0.5918
                            word 1-2gram + LinearSVC       0.5893           0.0171       0.5895
                          baseline: stratified guess       0.3369    

Read the two tables side by side and they disagree about features, which is the
point:

- **Sentiment**: word → char → word+char → word+char+surface climbs monotonically.
  The views are complementary, so we pay for all three.
- **Topic**: the character block *alone* wins, and adding the word block **costs**
  10 macro-F1 points. A word tokeniser cannot see `app` inside `happy`. This is
  Section 2.2 resurfacing as a model-selection result.

The winning character range for topic is **2–3** — the length of `ui`, `ban`, `app`,
`bug`. The hyperparameter that wins tells you what the label is made of.

### How much would a duplicate-blind split have invented?

In [11]:
import joblib
from config import MODEL_BUNDLE

bundle = joblib.load(MODEL_BUNDLE)
for task, leak in bundle["leakage"].items():
    print(f"{task:10s} grouped {leak['grouped_by_text_f1_macro']:.4f}  |  "
          f"random-row {leak['random_row_split_f1_macro']:.4f}  |  "
          f"inflation +{leak['inflation']:.4f}")

sentiment  grouped 0.6227  |  random-row 0.6790  |  inflation +0.0562
topic      grouped 0.9171  |  random-row 0.9204  |  inflation +0.0034


## 6. Held-out evaluation

20% of unique posts, stratified, untouched by every step above and scored exactly
once.

In [12]:
from sklearn.metrics import classification_report
from dataio import make_split
from config import TASKS

for task, col in TASKS.items():
    split = make_split(df, task)
    model = bundle["models"][task]
    y_true = split.test[col].to_numpy()
    y_pred = model.predict(split.test.post_text.to_numpy())
    print(f"=== {task}  ({bundle['selected'][task]})")
    print(classification_report(y_true, y_pred, zero_division=0))

=== sentiment  (word + char 3-5gram + surface + LinearSVC)
              precision    recall  f1-score   support

    Negative       0.65      0.69      0.67       597
     Neutral       0.54      0.55      0.55       590
    Positive       0.69      0.64      0.67       602

    accuracy                           0.63      1789
   macro avg       0.63      0.63      0.63      1789
weighted avg       0.63      0.63      0.63      1789



=== topic  (char 2-3gram + LinearSVC)
                      precision    recall  f1-score   support

    Account_Security       1.00      0.88      0.94        26
Community_Discussion       0.99      1.00      0.99      1528
    Feature_Feedback       0.93      0.84      0.88        62
    Technical_Issues       0.97      0.94      0.96       163

            accuracy                           0.99      1779
           macro avg       0.97      0.92      0.94      1779
        weighted avg       0.99      0.99      0.99      1779



In [13]:
metrics = json.loads((REPORTS / "metrics.json").read_text(encoding="utf-8"))
rows = []
for task in TASKS:
    b = metrics["tasks"][task]
    rows.append({"task": task, "macro_F1": b["macro_f1"], "accuracy": b["accuracy"],
                 "weighted_F1": b["weighted_f1"], "kappa": b["cohen_kappa"],
                 "MCC": b["matthews_corrcoef"]})
pd.DataFrame(rows).round(4)

,task,macro_F1,accuracy,weighted_F1,kappa,MCC
0,sentiment,0.6263,0.6266,0.6267,0.440,0.4403
1,topic,0.9426,0.9859,0.9857,0.943,0.9434


## 7. Error analysis

### Sentiment — are the mistakes confident?

In [14]:
b = metrics["tasks"]["sentiment"]["error_profile"]
print(f"mean confidence when right : {b['mean_conf_correct']:.3f}")
print(f"mean confidence when wrong : {b['mean_conf_wrong']:.3f}")
print(f"errors made above 0.8 conf : {b['share_of_errors_above_0_8_conf']:.1%}")
print()
err = pd.read_csv(REPORTS / "errors_sentiment.csv")
print("highest-confidence mistakes:")
for _, r in err.head(5).iterrows():
    print(f"  [{r.actual} -> {r.predicted}, p={r.confidence:.2f}] {str(r.post_text)[:84]}")

mean confidence when right : 0.684
mean confidence when wrong : 0.593
errors made above 0.8 conf : 5.2%

highest-confidence mistakes:
  [Neutral -> Negative, p=0.91] "If it weren't for Christians, the Aztecs would never have stopped killing babies (l
  [Positive -> Neutral, p=0.91] Josh Hamilton hit a two run homer in 2011 WS in the 10th with a sports hernia. If an
  [Negative -> Positive, p=0.91] Hot Mess Monday! I'm feeling it! #runninglate #crazyday #blah #happymonday #lovemyli
  [Positive -> Neutral, p=0.88] In rugby news\u002c @user are winning 26-5 at Cobham\u002c with the four-try bonus p
  [Neutral -> Positive, p=0.88] "Should've went to watch Paper Towns first, I'll just go see it tomorrow"


The separation between the two confidences is what makes a confidence threshold a
usable routing control: the engine can hand its hard cases to a human instead of
guessing.

The named failure modes — idiomatic negation (`can't wait`), context-dependent
polarity (sports and politics), the Neutral boundary on reporting-voice posts, and
sarcasm — are worked through in the technical report.

### Topic — the errors are rare-trigger recovery failures

In [15]:
rd = metrics["tasks"]["topic"]["rule_diagnosis"]
print(f"error rate, no trigger present     : {rd['error_rate_no_trigger']:.1%}")
print(f"error rate, trigger in >= 60 posts : {rd['error_rate_common_trigger']:.1%}")
print(f"error rate, trigger in <  60 posts : {rd['error_rate_rare_trigger']:.1%}")
print()
print(f"learned model held-out accuracy    : {rd['model_accuracy']:.4f}")
print(f"recovered rule held-out accuracy   : {rd['rule_accuracy']:.4f}")

error rate, no trigger present     : 0.2%
error rate, trigger in >= 60 posts : 1.4%
error rate, trigger in <  60 posts : 55.9%

learned model held-out accuracy    : 0.9859
recovered rule held-out accuracy   : 1.0000


The whole gap is rare-trigger recovery. The topic task has no modelling problem
left to solve — it has a **labelling** problem. More data or a bigger model would
converge on the same rule and would all be learning an artefact.

**Recommendation:** re-annotate `topic_category` before deploying it for routing. The
pipeline here transfers unchanged, because nothing in it was tuned to the rule.

## 8. Inference

In [16]:
from predict import score

score([
    "the app keeps crashing right after the latest update, fix this please",
    "absolutely loved the new dark mode, best thing you've shipped all year",
    "Parliament meets on Thursday to discuss the budget",
    "someone tried to log into my account from another country",
])

,post_text,sentiment_pred,sentiment_confidence,topic_pred,topic_confidence,topic_rule,topic_model_agrees_with_rule
0,the app keeps crashing right after the latest ...,Negative,0.4847,Technical_Issues,1.0000,Technical_Issues,True
1,"absolutely loved the new dark mode, best thing...",Positive,0.8646,Feature_Feedback,0.5105,Feature_Feedback,True
2,Parliament meets on Thursday to discuss the bu...,Neutral,0.5967,Community_Discussion,0.9805,Community_Discussion,True
3,someone tried to log into my account from anot...,Negative,0.4218,Account_Security,0.8822,Account_Security,True


---

### Deliverables produced by this pipeline

| Deliverable | File |
|---|---|
| NLP model notebook | this file |
| Trained models | `round2/models/social_engine_nlp_team_se7en.pkl` |
| Evaluation metrics report | `round2/reports/Evaluation_Metrics_Report_Team_SE7EN.pdf` |
| Technical report | `round2/reports/Round2_Technical_Report_Team_SE7EN.pdf` |

Rebuild everything from the raw CSV with `python round2/run_round2.py`.

**Team SE7EN** — Tanmay Singh · Panshul Arora